In [1]:
import os
import cv2
import math
import mediapipe as mp
import pandas as pd
import numpy as np
from collections import defaultdict

In [2]:
TRAIN_DIR = "../data/alphabet/raw/train"
TEST_DIR = "../data/alphabet/raw/test"

OUTPUT_TRAIN_CSV = "../data/alphabet/landmarks/train_landmarks_normalized.csv"
OUTPUT_TEST_CSV = "../data/alphabet/landmarks/test_landmarks_normalized.csv"

In [3]:
mp_hands = mp.solutions.hands

In [4]:
def preprocess_image_for_mediapipe(image, pad=80, target_size=512):
    if image is None:
        return None

    image = cv2.copyMakeBorder(
        image,
        pad, pad, pad, pad,
        borderType=cv2.BORDER_CONSTANT,
        value=(255, 255, 255)
    )

    image = cv2.resize(image, (target_size, target_size))
    return image

In [5]:
def normalize_landmarks(landmarks):
    wrist = landmarks[0]

    shifted = []
    for x, y, z in landmarks:
        shifted.append((x - wrist[0], y - wrist[1], z - wrist[2]))

    max_dist = 0.0
    for x, y, z in shifted:
        dist = math.sqrt(x**2 + y**2 + z**2)
        if dist > max_dist:
            max_dist = dist

    if max_dist == 0:
        return None

    normalized = []
    for x, y, z in shifted:
        normalized.extend([x / max_dist, y / max_dist, z / max_dist])

    return normalized

In [6]:
def extract_landmarks_from_image(image_path, hands):
    image = cv2.imread(image_path)
    if image is None:
        return None

    candidates = []

    # original
    candidates.append(image)

    # smaller padding versions
    for pad in [0, 10, 20, 40]:
        padded = cv2.copyMakeBorder(
            image, pad, pad, pad, pad,
            borderType=cv2.BORDER_CONSTANT,
            value=(255, 255, 255)
        )
        candidates.append(padded)

    # flipped versions too
    candidates += [cv2.flip(img, 1) for img in candidates]

    for img in candidates:
        img = cv2.resize(img, (512, 512))
        rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)

        if results.multi_hand_landmarks:
            coords = [(lm.x, lm.y, lm.z) for lm in results.multi_hand_landmarks[0].landmark]
            normalized = normalize_landmarks(coords)
            if normalized is not None:
                return normalized

    return None

In [7]:
def process_dataset(input_dir, split_name):
    data = []
    failed_files = []
    stats = defaultdict(lambda: {"total": 0, "kept": 0, "failed": 0})

    with mp_hands.Hands(
        static_image_mode=True,
        max_num_hands=1,
        min_detection_confidence=0.3
    ) as hands:

        for label in sorted(os.listdir(input_dir)):
            label_path = os.path.join(input_dir, label)

            if not os.path.isdir(label_path):
                continue

            print(f"Processing {split_name} label: {label}")

            for file_name in os.listdir(label_path):
                file_path = os.path.join(label_path, file_name)
                stats[label]["total"] += 1

                features = extract_landmarks_from_image(file_path, hands)

                if features is not None:
                    row = features + [label, file_path, split_name]
                    data.append(row)
                    stats[label]["kept"] += 1
                else:
                    stats[label]["failed"] += 1
                    failed_files.append({
                        "label": label,
                        "file_path": file_path,
                        "split": split_name
                    })

    columns = []
    for i in range(21):
        columns.extend([f"x{i}", f"y{i}", f"z{i}"])
    columns += ["label", "file_path", "split"]

    df = pd.DataFrame(data, columns=columns)
    stats_df = pd.DataFrame(stats).T.reset_index().rename(columns={"index": "label"})
    failed_df = pd.DataFrame(failed_files)

    return df, stats_df, failed_df

In [8]:
train_df, train_stats, train_failed = process_dataset(TRAIN_DIR, "train")
test_df, test_stats, test_failed = process_dataset(TEST_DIR, "test")

Processing train label: A


d:\Fontys\Semester 4 - ML\Sign Language Recognition\Project\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing train label: B
Processing train label: C
Processing train label: D
Processing train label: E
Processing train label: F
Processing train label: G
Processing train label: H
Processing train label: I
Processing train label: K
Processing train label: L
Processing train label: M
Processing train label: N
Processing train label: O
Processing train label: P
Processing train label: Q
Processing train label: R
Processing train label: S
Processing train label: T
Processing train label: U
Processing train label: V
Processing train label: W
Processing train label: X
Processing train label: Y
Processing test label: A


d:\Fontys\Semester 4 - ML\Sign Language Recognition\Project\venv\Lib\site-packages\google\protobuf\symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Processing test label: B
Processing test label: C
Processing test label: D
Processing test label: E
Processing test label: F
Processing test label: G
Processing test label: H
Processing test label: I
Processing test label: K
Processing test label: L
Processing test label: M
Processing test label: N
Processing test label: O
Processing test label: P
Processing test label: Q
Processing test label: R
Processing test label: S
Processing test label: T
Processing test label: U
Processing test label: V
Processing test label: W
Processing test label: X
Processing test label: Y


In [9]:
os.makedirs("../data/alphabet/landmarks", exist_ok=True)

train_df.to_csv(OUTPUT_TRAIN_CSV, index=False)
test_df.to_csv(OUTPUT_TEST_CSV, index=False)

print("Saved:")
print(OUTPUT_TRAIN_CSV)
print(OUTPUT_TEST_CSV)

Saved:
../data/alphabet/landmarks/train_landmarks_normalized.csv
../data/alphabet/landmarks/test_landmarks_normalized.csv


In [10]:
train_failed.head(20)

,label,file_path,split
0,A,../data/alphabet/raw/train\A\Image_1685009090....,train
1,A,../data/alphabet/raw/train\A\Image_1685009110....,train
2,A,../data/alphabet/raw/train\A\Image_1685009112....,train
3,A,../data/alphabet/raw/train\A\Image_1685009115....,train
4,A,../data/alphabet/raw/train\A\Image_1685009120....,train
5,A,../data/alphabet/raw/train\A\Image_1685009124....,train
6,A,../data/alphabet/raw/train\A\Image_1685009126....,train
7,A,../data/alphabet/raw/train\A\Image_1685009127....,train
8,A,../data/alphabet/raw/train\A\Image_1685009136....,train
9,A,../data/alphabet/raw/train\A\Image_1685009138....,train


In [11]:
train_stats["keep_rate"] = train_stats["kept"] / train_stats["total"]
test_stats["keep_rate"] = test_stats["kept"] / test_stats["total"]

train_stats.sort_values("kept", ascending=False)

,label,total,kept,failed,keep_rate
0,A,447,390,57,0.872483
6,G,435,350,85,0.804598
7,H,432,311,121,0.719907
23,Y,438,279,159,0.636986
9,K,455,274,181,0.602198
18,T,414,268,146,0.647343
14,P,438,267,171,0.609589
15,Q,449,259,190,0.576837
12,N,421,230,191,0.546318
4,E,441,229,212,0.519274


In [12]:
test_stats.sort_values("kept", ascending=False)

,label,total,kept,failed,keep_rate
6,G,75,57,18,0.760000
12,N,75,56,19,0.746667
18,T,75,56,19,0.746667
7,H,75,55,20,0.733333
10,L,75,48,27,0.640000
4,E,75,44,31,0.586667
15,Q,75,42,33,0.560000
5,F,75,42,33,0.560000
0,A,75,40,35,0.533333
14,P,75,39,36,0.520000
